[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session4/notebooks/03_build_html.ipynb)

# session4 · 03 · 강의자료 HTML 빌더 (수업 전에 1회 실행)

- **이 노트북에서 하는 것**: `01_train_lab` 을 실행해 `outputs/` 를 채우고, 강의 템플릿에 그림을 **base64 인라인** → **단일 파일** `session4_lecture.html` 생성.
- **입력**: `session4/session4_lecture_template.html`(claude.ai 제공) + 01 실행 결과.
- **출력**: `outputs/session4_lecture.html` (외부 연결 없는 자체완결 — 다운로드해서 사용).

> ⚠️ **파이프라인**입니다. **셀 1 → 2 → 3 → 4 → 5 순서대로** 실행하세요.
> ⚠️ **셀 3은 01 의 실험 전체(무료 T4 기준 8~11분)를 포함**합니다. **GPU 런타임**으로 실행하세요.
> ⚠️ **`02_pose_demo` 는 이 빌더를 끝낸 뒤 맨 마지막에** 실행하세요(mediapipe 가 런타임 패키지 구성을 바꿀 수 있습니다).
> `02` 의 산출물(`04_hand.png`·`04_pose.png`)은 **개인 얼굴·손 사진이라 HTML 에 주입하지 않습니다** — 라이브에서 화면으로만 봅니다.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (01 과 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session4"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session4/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# 셀 2 · 의존성 설치 + 실행 도구 확인 (nbconvert)
import os, sys, subprocess

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
print("requirements.txt 설치 중...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

try:
    import nbconvert
    print("nbconvert:", nbconvert.__version__)
except ImportError:
    print("nbconvert 미설치 -> 설치")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbconvert"], check=False)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (실험이 매우 느려집니다)")

In [ ]:
# 셀 3 · 01_train_lab 실행 (같은 VM) — 실험 전체를 포함해 8~11분 걸립니다.
import os, sys, subprocess

NB = "01_train_lab.ipynb"
src = os.path.join("notebooks", NB)
if not os.path.exists(src):
    raise RuntimeError("노트북이 없습니다: " + src)

exec_dir = os.path.abspath(os.path.join("outputs", "_executed"))   # 실행본 사본(비커밋)
os.makedirs(exec_dir, exist_ok=True)

print("=== 실행:", NB, "(실험 8회 포함 — 셀당 최대 1800초) ===")
cmd = [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute",
       "--ExecutePreprocessor.timeout=1800",
       "--output-dir", exec_dir, "--output", NB.replace(".ipynb", "_executed.ipynb"), src]
try:
    cp = subprocess.run(cmd, check=False)
    status = "성공" if cp.returncode == 0 else "실패(exit %d)" % cp.returncode
except Exception as e:
    status = "예외: %s" % e
print("->", status)

EXPECTED = ["04_dataset.png", "04_lr_compare.png",
            "04_lr_1e-1.png", "04_lr_1e-2.png", "04_lr_1e-3.png", "04_lr_1e-4.png", "04_lr_1e-6.png",
            "04_overfit.png", "04_aug.png", "04_confusion.png", "04_tradeoff.png"]
OPTIONAL = ["04_datasize.png"]      # 셀 6.5(추가 실험) 산출물 — 템플릿에 슬롯이 있으면 함께 주입된다
present = [f for f in EXPECTED if os.path.exists(os.path.join("outputs", f))]
missing = [f for f in EXPECTED if f not in present]
print("--- outputs/ 그림 점검 ---  있음 %d / %d" % (len(present), len(EXPECTED)))
if missing:
    print("  누락:", missing, "(빌드는 누락 자리표시로 진행됩니다)")
for f in OPTIONAL:
    print("  (선택) %-18s : %s" % (f, "있음" if os.path.exists(os.path.join("outputs", f)) else "없음"))
print("  runs.json :", "있음" if os.path.exists(os.path.join("outputs", "runs.json")) else "없음")

In [ ]:
# 셀 4 · HTML 조립 — 템플릿의 <img data-embed="..."> 슬롯에만 base64 src 를 주입
# (템플릿의 내용/구조/CSS 는 건드리지 않는다. img 태그에 src 속성만 추가한다.
#  <code>data-embed="..."</code> 같은 '설명 텍스트'는 img 가 아니므로 절대 건드리지 않는다.)
import os, re, base64

TEMPLATE = "session4_lecture_template.html"      # session4/ 루트 기준 (claude.ai 제공)

# 02_pose_demo 산출물은 개인 얼굴·손 사진이라 레포·HTML 에 넣지 않는다(라이브에서 화면으로만).
BLOCKED = {"04_hand.png", "04_pose.png"}

if not os.path.exists(TEMPLATE):
    # 템플릿은 claude.ai 가 작성해 session4/ 에 커밋한다. 없으면 빌더는 안내만 하고 조용히 끝낸다.
    print("=" * 62)
    print(" 강의 템플릿이 아직 없습니다:", TEMPLATE)
    print("=" * 62)
    print(" 이 파일은 claude.ai 가 작성해 session4/ 에 커밋합니다.")
    print(" 템플릿을 받아 session4/ 에 두고 이 셀(셀 4)을 다시 ▶ 실행하세요.")
    print()
    print(" 참고: 템플릿은 그림 자리에 다음처럼 빈 img 슬롯을 두면 됩니다.")
    print('   <img data-embed="04_lr_compare.png" alt="learning rate comparison">')
    print(" 주입 대상 그림 11개:")
    for f in ["04_dataset.png", "04_lr_compare.png",
              "04_lr_1e-1.png", "04_lr_1e-2.png", "04_lr_1e-3.png", "04_lr_1e-4.png", "04_lr_1e-6.png",
              "04_overfit.png", "04_aug.png", "04_confusion.png", "04_tradeoff.png"]:
        print("   -", f, "" if os.path.exists(os.path.join("outputs", f)) else "  (아직 outputs/ 에 없음)")
    print(" (선택) 04_datasize.png — 셀 6.5 추가 실험. 슬롯을 두면 함께 주입됩니다.")
    print(" (제외) 04_hand.png / 04_pose.png — 개인 사진이라 주입하지 않습니다.")
else:
    with open(TEMPLATE, encoding="utf-8") as _f:
        html = _f.read()

    embedded, missing, blocked = [], [], []
    IMG_RE = re.compile(r'<img\b[^>]*?\bdata-embed="([^"]+)"[^>]*?>', re.IGNORECASE | re.DOTALL)

    def inject(m):
        tag, fname = m.group(0), m.group(1)
        if fname in BLOCKED:
            blocked.append(fname)
            return tag                                    # 개인 사진: 주입하지 않고 자리표시 유지
        path = os.path.join("outputs", fname)
        if not os.path.exists(path):
            missing.append(fname)
            return tag                                    # 누락: 원본 그대로(템플릿 자리표시 노출)
        mime = "image/gif" if fname.lower().endswith(".gif") else "image/png"
        with open(path, "rb") as _img:
            b64 = base64.b64encode(_img.read()).decode("ascii")
        embedded.append(fname)
        return tag[:4] + ' src="data:%s;base64,%s"' % (mime, b64) + tag[4:]   # '<img' 바로 뒤에 src 삽입

    out_html = IMG_RE.sub(inject, html)
    os.makedirs("outputs", exist_ok=True)
    OUT = os.path.join("outputs", "session4_lecture.html")
    with open(OUT, "w", encoding="utf-8") as f:
        f.write(out_html)

    n_img_slot = len(IMG_RE.findall(html))
    n_mentions = len(re.findall(r'data-embed="', html))
    print("img 슬롯:", n_img_slot, "개  ->  임베드:", len(embedded), "| 누락:", len(missing), "| 제외:", len(blocked))
    if embedded:
        print("  임베드됨:", embedded)
    if missing:
        print("  누락(자리표시 유지):", missing)
    if blocked:
        print("  제외(개인 사진 — 의도된 동작):", blocked)
    if n_mentions > n_img_slot:
        print("  (참고) img 가 아닌 data-embed 언급", n_mentions - n_img_slot,
              "건은 템플릿의 설명 텍스트입니다 -> 건드리지 않음")
    print("저장:", OUT)

In [ ]:
# 셀 5 · 검증 / 다운로드 안내
import os

OUT = os.path.join("outputs", "session4_lecture.html")
print("=" * 56)
print(" session4 · 강의 HTML 빌드 검증")
print("=" * 56)
if os.path.exists(OUT):
    print(" - 최종 파일 :", OUT)
    print(" - 크기      : %.2f MB" % (os.path.getsize(OUT) / 1e6))
    print("=" * 56)
    print("이 파일을 다운로드해 브라우저로 열면, 외부 연결 없이 단독으로 보이는 강의 자료입니다.")
    try:
        import google.colab  # noqa: F401
        print("(코랩) 좌측 파일창에서 outputs/session4_lecture.html 우클릭 -> 다운로드,")
        print("       또는 새 셀에: from google.colab import files; files.download('%s')" % OUT)
    except ImportError:
        print("(로컬) 위 경로의 파일을 브라우저로 열면 됩니다.")
else:
    print(" [주의] 최종 HTML이 없습니다.")
    print("        - 템플릿(session4_lecture_template.html)이 아직 없으면 정상입니다 — 셀 4의 안내를 보세요.")
    print("        - 템플릿이 있는데도 없다면 셀 1->2->3->4 를 순서대로 실행했는지 확인하세요.")